# Deep Research Agent

基于 ReAct 模式的多轮检索问答 Agent, 在 BrowseComp-Plus 语料上运行. 

## 1. 环境配置

In [ ]:
import sys
import json
from pathlib import Path

# 项目根目录
PROJECT_ROOT = Path(".").resolve()
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# vLLM 服务配置
BASE_URL = "http://127.0.0.1:8000/v1"
MODEL_NAME = "qwen_auto"

# BM25 索引路径
INDEX_PATH = str(PROJECT_ROOT / "browsecomp_bm25.db")

# 数据集路径
DATASET_PATH = str(PROJECT_ROOT / "browsecomp_plus_hard50.jsonl")

# 输出路径
OUTPUT_DIR = PROJECT_ROOT / "runs"
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. 初始化组件

In [ ]:
from agent.browsecomp_searcher import BrowseCompBM25Searcher
from agent.vllm_client import VLLMClient
from agent.react_agent import DeepResearchAgent
from agent.dataset_utils import load_jsonl

# 初始化检索器
searcher = BrowseCompBM25Searcher(index_path=INDEX_PATH)
print(f"Searcher initialized: {searcher.search_type}")

# 初始化 vLLM 客户端
client = VLLMClient(base_url=BASE_URL)
print(f"VLLMClient initialized: {BASE_URL}")

# 初始化 Agent (V9: gap-driven ReAct)
agent = DeepResearchAgent(
    client=client,
    searcher=searcher,
    model_name=MODEL_NAME,
    max_rounds=10,
    search_top_k=20,
    snippet_max_chars=600,
    max_recent_rounds=3,
    temperature=0.0,
)
print(f"Agent initialized: max_rounds={agent.max_rounds}, temperature={agent.temperature}")

## 3. 加载数据集

In [ ]:
dataset = load_jsonl(DATASET_PATH)
print(f"Loaded {len(dataset)} queries")
print(f"Sample: query_id={dataset[0].get('query_id')}, question={dataset[0].get('query', '')[:100]}...")

## 4. 运行 Agent(小规模测试)

In [ ]:
# 先跑前 3 条测试
test_results = []
for row in dataset[:3]:
    query_id = row["query_id"]
    question = row.get("query", "")
    print(f"\n{'='*60}")
    print(f"Query {query_id}: {question[:80]}...")
    
    result = agent.run(question, verbose=True)
    result["query_id"] = query_id
    test_results.append(result)
    
    print(f"  Predicted: {result['predicted_answer'][:80]}")
    print(f"  Gold: {row.get('answer', 'N/A')[:80]}")
    print(f"  Status: {result['status']}")

## 5. 运行完整评估

In [ ]:
# 运行全部查询
all_results = []
for i, row in enumerate(dataset):
    query_id = row["query_id"]
    question = row.get("query", "")
    print(f"\n[{i+1}/{len(dataset)}] Query {query_id}: {question[:60]}...")
    
    result = agent.run(question, verbose=True)
    result["query_id"] = query_id
    all_results.append(result)
print('all query done.')

## 6. 生成提交文件

In [ ]:
# 生成 submission.jsonl
submission_path = OUTPUT_DIR / "submission.jsonl"
with submission_path.open("w", encoding="utf-8") as f:
    for result in all_results:
        entry = {
            "query_id": result["query_id"],
            "predicted_answer": result["predicted_answer"],
            "status": result["status"],
            "messages": result["messages"],
        }
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Submission saved to: {submission_path}")
print(f"Total queries: {len(all_results)}")

## 7. 自动评估

In [ ]:
from agent.eval import run_evaluation

summary, details = run_evaluation(
    submission_path=str(submission_path),
    dataset_path=DATASET_PATH,
    model_name=MODEL_NAME,
    base_url=BASE_URL,
    output_path=str(OUTPUT_DIR / "eval_results.jsonl"),
)

print(f"\nAccuracy: {summary['accuracy']:.2%}")
print(f"Correct: {summary['correct']}/{summary['total_queries']}")

## 8. 生成最终提交文件(含分数命名)

In [ ]:
# 根据评估结果生成最终提交文件
acc_value = summary['accuracy']
# 分数中小数点用下划线: 0.12 → 12
acc_str = str(int(acc_value * 100))  # e.g. "12"

STUDENT_ID = "231300081"  
STUDENT_NAME = "刘鹤"       

final_filename = f"{STUDENT_ID}-{STUDENT_NAME}-submission-{acc_str}.jsonl"
final_path = OUTPUT_DIR / final_filename

# 复制 submission.jsonl 到最终文件名
import shutil
shutil.copy(submission_path, final_path)

print(f"Final submission: {final_path}")
print(f"Accuracy: {acc_value:.2%}")